[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wildertrek/catcher-in-the-cache/blob/main/notebooks/05_cache_map.ipynb)

# 05: SCPI as the Cache-Membership Gauge

> *"People always think something's all true."* ~ Holden Caulfield

SCPI is the explicit retrieval method (M6). It cannot *measure* a novel character, but the
*distance* it reports is the useful signal. This notebook uses it as a **map of the cache**:

- **#1 Am I on the map?** A character's similarity to known canonical characters flags whether
  an LLM rating is a replayed prior (in-cache) or confabulation (out-of-cache).
- **#2 Where are the blank spots?** The trait-space regions no canonical character occupies are
  exactly where retrieval must fail and where a cache-free regressor needs training data.

All values precomputed (cache scores, HEXACO, t-SNE) into `cache_map_viz_data.json`; this notebook just renders.

## Setup

**In Colab, run the cell below first.** It clones the companion repository and installs dependencies (~30 s), then changes into `notebooks/` so the relative paths in this notebook resolve. Run locally, the cell is a no-op.

In [1]:
# Colab setup: clone the companion repo + install dependencies (no-op when run locally).
import sys, os, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("catcher-in-the-cache"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Wildertrek/catcher-in-the-cache.git"], check=True)
    os.chdir("catcher-in-the-cache/notebooks")
    subprocess.run(["pip", "install", "-q", "-r", "../requirements.txt"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())

In [2]:
%pip install -q plotly numpy
import json, numpy as np, plotly.graph_objects as go
from pathlib import Path
p=Path('paper_artifacts/pivot6_hexaco_atlas/cache_map_viz_data.json')
if not p.exists(): p=Path('../paper_artifacts/pivot6_hexaco_atlas/cache_map_viz_data.json')
D=json.loads(p.read_text()); pts=D['points']; TPL='plotly_white'
canon=[q for q in pts if q['type']=='canonical']; synth=[q for q in pts if q['type']=='synth']
print(f"{len(canon)} canonical + {len(synth)} synth characters")


Note: you may need to restart the kernel to use updated packages.


1981 canonical + 20 synth characters


## 1. Where the synthetic characters reside

t-SNE of the 3072-d character embeddings. Gray = ~2000 canonical characters; red = the 20
synthetics. They land as their own island at the edge: genuinely out-of-distribution.

In [3]:
fig=go.Figure()
fig.add_trace(go.Scattergl(x=[q['x'] for q in canon],y=[q['y'] for q in canon],mode='markers',
  name='canonical (~2000)',marker=dict(size=4,color='#b2bec3',opacity=0.55),
  text=[f"{q['name']} (cache {q['cache']})" for q in canon],hoverinfo='text'))
fig.add_trace(go.Scatter(x=[q['x'] for q in synth],y=[q['y'] for q in synth],mode='markers+text',
  name='synthetic (20)',marker=dict(size=11,color='#D63031',line=dict(width=1,color='#fff')),
  text=[q['name'] for q in synth],textposition='top center',textfont=dict(size=8),
  hovertext=[f"{q['name']} (cache {q['cache']})" for q in synth],hoverinfo='text'))
fig.update_layout(template=TPL,height=620,title='Synthetic characters sit apart from the canonical cloud (t-SNE)',
  xaxis_title='t-SNE 1',yaxis_title='t-SNE 2',legend=dict(orientation='h',y=-0.12))
fig.show()


## 2. Am I on the map? (the cache-membership gauge, #1)

Each character's max similarity to the known canonical set. Canonical characters cluster high
(median 0.87); every synthetic falls below the canonical 5th percentile (synthetic median 0.41,
zero overlap with the canonical distribution). A ~0.55 threshold flags out-of-cache characters,
route those to measurement, do not trust an LLM rating. (Corpus membership, not fame.)

In [4]:
g=D['gauge']
fig=go.Figure()
fig.add_trace(go.Histogram(x=g['canon_scores'],name='canonical',histnorm='probability',marker_color='#0984E3',opacity=0.7,nbinsx=40))
fig.add_trace(go.Histogram(x=g['synth_scores'],name='synthetic',histnorm='probability',marker_color='#D63031',opacity=0.8,nbinsx=20))
fig.add_vline(x=g['threshold'],line_dash='dash',line_color='#2d3436',annotation_text=f"route threshold {g['threshold']}")
fig.add_vline(x=g['canon_p5'],line_dash='dot',line_color='#0984E3',annotation_text=f"canonical p5 {g['canon_p5']}",annotation_position='top left')
fig.update_layout(template=TPL,height=460,barmode='overlay',title='Cache-membership gauge: in-corpus vs out',
  xaxis_title='max cosine similarity to known canonical characters',yaxis_title='proportion',legend=dict(orientation='h',y=-0.2))
fig.show()

# RQ3.2 discriminability: AUC + false-flag at recall 1.0
import numpy as _np
_c=_np.array(g['canon_scores']); _s=_np.array(g['synth_scores']); _thr=g['threshold']
_auc=float((_c[:,None] > _s[None,:]).mean())
_recall=float((_s < _thr).mean()); _ff=float((_c < _thr).mean())
print(f'Gauge discriminability: AUC = {_auc:.3f}')
print(f'  at the {_thr} threshold: recall on out-of-cache = {_recall:.0%} (all synthetics caught), '
      f'false-flag on canonical = {_ff:.1%}')


Gauge discriminability: AUC = 0.996
  at the 0.55 threshold: recall on out-of-cache = 100% (all synthetics caught), false-flag on canonical = 2.3%


## 3. Where are the blank spots? (the coverage holes, #2)

Honesty-Humility vs HEXACO-Agreeableness. Canonical characters (gray) cluster on the fused
diagonal (r = +0.86). The two decoupled corners (honest-harsh, dishonest-kind) are **empty of
canonical characters** but populated by synthetics (red, r = -0.74). Retrieval has nothing to
return in the shaded corners, exactly where the cache-free regressor must learn.

In [5]:
ch=[q for q in canon if q['H'] is not None]
fig=go.Figure()
for (x0,x1,y0,y1) in [(0.3,1,-1,-0.3),(-1,-0.3,0.3,1)]:
  fig.add_shape(type='rect',x0=x0,x1=x1,y0=y0,y1=y1,fillcolor='#ffeaa7',opacity=0.45,line_width=0,layer='below')
fig.add_trace(go.Scatter(x=[q['H'] for q in ch],y=[q['A_HEX'] for q in ch],mode='markers',name=f'canonical ({len(ch)})',
  marker=dict(size=7,color='#636e72',opacity=0.7),text=[q['name'] for q in ch],hoverinfo='text'))
fig.add_trace(go.Scatter(x=[q['H'] for q in synth],y=[q['A_HEX'] for q in synth],mode='markers',name='synthetic (20)',
  marker=dict(size=11,color='#D63031',line=dict(width=1,color='#fff')),text=[q['name'] for q in synth],hoverinfo='text'))
fig.add_annotation(x=0.62,y=-0.62,text='honest-harsh<br>(empty of canon)',showarrow=False,font=dict(size=10))
fig.add_annotation(x=-0.62,y=0.62,text='dishonest-kind<br>(empty of canon)',showarrow=False,font=dict(size=10))
fig.update_layout(template=TPL,height=560,title='Coverage holes: canonical fusions (r=+0.86); the decoupled corners are empty',
  xaxis_title='Honesty-Humility (H)',yaxis_title='HEXACO-Agreeableness (A_HEX)',
  xaxis_range=[-1,1],yaxis_range=[-1,1],legend=dict(orientation='h',y=-0.18))
fig.show()


## 4. Every method fusions except the truth

r(H, A_HEX) across the 20 synthetic characters, who were *designed* anti-correlated. The LLM
raters, the LLM-trained regressor, and SCPI retrieval all return the fused (+) pattern; SCPI,
the purest retrieval, is the most extreme (r = +0.92, a full sign-flip of the designed −0.74).
Only the author-designed truth is decoupled.

In [6]:
s=D['summary']
labels=['author-designed<br>truth','LLM raters<br>(Mode 1)','LLM-trained<br>regressor (Mode 2)','SCPI retrieval<br>(M6 floor)']
vals=[s['designed_r'],s['llm_r'],s['regressor_r'],s['scpi_retrieval_r']]
cols=['#00b894','#fdcb6e','#e17055','#D63031']
fig=go.Figure(go.Bar(x=labels,y=vals,marker_color=cols,text=[f'{v:+.2f}' for v in vals],textposition='outside'))
fig.add_hline(y=0,line_color='#dfe6e9')
fig.update_layout(template=TPL,height=460,title='r(H, A_HEX) on 20 synthetic chars: retrieval flips the designed −0.74 to +0.92',
  yaxis_title='r(H, A_HEX)',yaxis_range=[-0.9,1],showlegend=False)
fig.show()


## Reading

- **Fig 1 + 2:** the synthetics are a separate island; the gauge separates in-corpus from
  out-of-corpus with zero overlap. SCPI similarity is a deployable trust check on LLM ratings.
- **Fig 3:** the canonical corpus has *no* characters in the decoupled corners, so retrieval
  cannot reach them; that void is the cache-free regressor's training target.
- **Fig 4:** retrieval returns the opposite of the designed structure, the cleanest statement
  of retrieval-not-measurement.

Map (SCPI) finds the holes and flags the edge; the cache-free regressor measures past it.
Artifacts: `cache_membership_and_coverage.json`, `synth_scpi_retrieval.json`, `cache_map_viz_data.json`.